In [24]:
import wandb
import pandas as pd
import numpy as np

api = wandb.Api()
runs = api.runs("xiaoxionglin-bernstein-center-freiburg/NEWymaze_norew_instr")

records = []
for run in runs:
    policies = {}
    for p in range(4):
        hist = run.history(keys=[f"{p}/custom/flexibility", f"{p}/len/len", f"{p}/custom/highrew_hit"], pandas=True)
        if not hist.empty:
            policies[p] = hist

            N = 20
            tail = hist.tail(N)

            flex_mean = tail[f"{p}/custom/flexibility"].mean()
            flex_abs_mean = tail[f"{p}/custom/flexibility"].abs().mean()
            flex_std = tail[f"{p}/custom/flexibility"].std()
            reward_mean = tail[f"{p}/custom/highrew_hit"].mean()

            early = hist[f"{p}/len/len"].head(N).mean()
            late = tail[f"{p}/len/len"].mean()
            shortened_ratio = late / early if early > 0 else np.nan

            records.append({
                "run_id": run.id, "name": run.name, "policy": p,
                "flex_mean": flex_mean, "flex_abs_mean": flex_abs_mean, "flex_std": flex_std,
                "shortened_ratio": shortened_ratio, "reward_mean": reward_mean,
            })

df = pd.DataFrame(records)

FLEX_TOL, REWARD_TOL = 0.15, 5
mask = (
    (df["flex_abs_mean"].between(1 - FLEX_TOL, 1)) &
    (df["flex_std"] < 0.2) &
    (df["shortened_ratio"] < 0.7) &
    (df["reward_mean"].between(60 - REWARD_TOL, 60 + REWARD_TOL))
)

selected = df[mask].sort_values("reward_mean", ascending=False)
selected.to_csv("selected_runs.csv", index=False)

KeyError: 'flex_abs_mean'

In [23]:
# Step 0: inspect keys on one run before looping over all 180
probe = runs[0]
probe_hist = probe.history(pandas=True)
print("Available columns:", [c for c in probe_hist.columns if "flex" in c.lower()])
finished_runs = api.runs(
    "xiaoxionglin-bernstein-center-freiburg/NEWymaze_norew_instr",
    filters={"state": "crashed"}
)

records = []
skipped = []
for run in finished_runs:
    hist = run.history(pandas=True, samples=1000)  # no key filter, no dropna surprises
    if hist.empty:
        skipped.append((run.id, "empty history"))
        continue

    flex_col = next((c for c in hist.columns if "flexibility" in c.lower()), None)
    if flex_col is None:
        skipped.append((run.id, "no flexibility column"))
        continue

    flex_series = hist[flex_col].dropna()
    ep_len_series = hist["len/len"].dropna() if "len/len" in hist.columns else pd.Series(dtype=float)
    reward_series = hist["highrew_hit"].dropna() if "highrew_hit" in hist.columns else pd.Series(dtype=float)

    if flex_series.empty:
        skipped.append((run.id, "flexibility all-NaN"))
        continue

    N = 20
    tail_flex = flex_series.tail(N)
    flex_abs_mean = tail_flex.abs().mean()
    flex_std = tail_flex.std()
    reward_mean = reward_series.tail(N).mean() if not reward_series.empty else np.nan

    if not ep_len_series.empty and len(ep_len_series) >= N:
        shortened_ratio = ep_len_series.tail(N).mean() / ep_len_series.head(N).mean()
    else:
        shortened_ratio = np.nan

    records.append({
        "run_id": run.id, "name": run.name,
        "flex_abs_mean": flex_abs_mean, "flex_std": flex_std,
        "shortened_ratio": shortened_ratio, "reward_mean": reward_mean,
    })

df = pd.DataFrame(records)
print("Built df with", len(df), "rows; skipped", len(skipped), "runs")

if not df.empty:
    FLEX_TOL, REWARD_TOL = 0.15, 5
    mask = (
        df["flex_abs_mean"].between(1 - FLEX_TOL, 1) &
        (df["flex_std"] < 0.2) &
        (df["shortened_ratio"] < 0.7) &
        df["reward_mean"].between(60 - REWARD_TOL, 60 + REWARD_TOL)
    )
    selected = df[mask].sort_values("reward_mean", ascending=False)
    selected.to_csv("selected_runs.csv", index=False)

CommError: Failed to execute API request: the service process is busy and did not respond in time.